**第12回：SOFR vs TONA ベーシス スワップ**

In [1]:
# Sofr data
sfSwRT= [('depo', '1d', 3.60),
      ('swap','1w',3.66154), ('swap','2w',3.68178), ('swap','3w',3.67395),
      ('swap','1m',3.6731),  ('swap','2m',3.6794),  ('swap','3m',3.67135),
      ('swap','4m',3.65985), ('swap','5m',3.65005), ('swap','6m',3.6309),
      ('swap','7m',3.6101),  ('swap','8m',3.59355), ('swap','9m',3.5725),
      ('swap','10m',3.55115),('swap','11m',3.5339), ('swap','12m',3.5188),
      ('swap','18m',3.43495),('swap','2y',3.4272),  ('swap','3y',3.4609),
      ('swap','4y',3.5215),  ('swap','5y',3.58825)]
# Tona data
tnSwRT = [('depo','1d', 0.728),
      ('swap','1w',0.7277),  ('swap','2w',0.72875), ('swap','3w',0.72875),
      ('swap','1m',0.72875), ('swap','2m',0.72956), ('swap','3m',0.7414),
      ('swap','4m',0.76665), ('swap','5m',0.79104), ('swap','6m',0.81747),
      ('swap','7m',0.84223), ('swap','8m',0.86397), ('swap','9m',0.8872),
      ('swap','10m',0.91146),('swap','11m',0.9338), ('swap','12m',0.9575),
      ('swap','15m',1.02),   ('swap','18m',1.0825), ('swap','2y',1.2025),
      ('swap','3y',1.3775),  ('swap','4y',1.51375), ('swap','5y',1.6275)]
# xccy Basis
xcyBSS = [('fxsw','1W',-9.43),('fxsw','1M',-44.85),('fxsw','2M',-81.51),
        ('fxsw','3M',-121.85),('fxsw','4M',-160.19),('fxsw','5M',-198.75),
        ('fxsw','6M',-234.62),('fxsw','9M',-337.59),('fxsw','12M',-433.70),
        ('bss','18m',-25.625),('bss','2y',-28.125), ('bss','3y',-31.375),
        ('bss','4y',-34.25),  ('bss','5y',-36.5)]

In [2]:
from myABBR import * ; import myUTIL as mu

trdDT = jDT(2026,1,20); calUJ=ql.JointCalendar(calUSf, calJP)
stlDT = adD(calUJ, trdDT,Tp2); setEvDT(trdDT)
# crv data
sfIX, sfCvOBJ, sfCvHDL, sfParRT = mu.makeSofrCurve(sfSwRT)
tnIX, tnCvOBJ, tnCvHDL, tnParRT = mu.makeTonaCurve(tnSwRT)
# xcBasis base: sofr
ujFX,  baseIX, othrIX, clltCv, baseCLLT, baseBSS, baseRST, cHelper, fxParRT=\
158.1528, sfIX,  tnIX,  sfCvHDL, True,     False,   True,      [],     []

for kk, tt, bb in xcyBSS:
  if kk == 'fxsw':
    cHelper.append(ql.FxSwapRateHelper( sQH(bb/100),sQH(ujFX),pD(tt),
                              Tp2,calUJ,mFLLW,EoMf,baseCLLT,sfCvHDL) )
    fxParRT.append(bb/100)
  if kk == 'bss':
    cHelper.append( ql.MtMCrossCurrencyBasisSwapRateHelper(
                  sQH(bb/10000),pD(tt),Tp2,calUJ,mFLLW,EoMf,
                  baseIX,othrIX,clltCv,baseCLLT,baseBSS,baseRST,frqQ))
    fxParRT.append(bb/10000)

jBsCvOBJ = ql.PiecewiseLogLinearDiscount(Tp0,calUJ,cHelper,dcA365)
jBsCvHDL = ql.YieldTermStructureHandle(jBsCvOBJ)

mu.showCvDT(jBsCvOBJ); dfDSP(mu.dfNodes(jBsCvOBJ,fxParRT,cmpd=CNT))

tradeDT:2026-01-20, settleDT:2026-01-20 

,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,0.601083%,1.00000000
1,2026-01-29,0.0247,-9.430000%,0.601083%,0.99985180
2,2026-02-24,0.0959,-44.850000%,0.578398%,0.99944553
3,2026-03-23,0.1699,-81.510000%,0.576547%,0.99902114
4,2026-04-22,0.2521,-121.850000%,0.569336%,0.99856599
10,2027-07-22,1.5014,-0.256250%,0.821466%,0.98774250
11,2028-01-24,2.0110,-0.281250%,0.915075%,0.98176649
12,2029-01-22,3.0082,-0.313750%,1.057407%,0.96869148
13,2030-01-22,4.0082,-0.342500%,1.165301%,0.95436619
14,2031-01-22,5.0082,-0.365000%,1.257487%,0.93896442


**元本固定型**

In [3]:
# 元本固定, JPY-Payer
effDT,   matDT,              usAMT,     spdRT,       stlDtCF     =\
stlDT, adY(calUJ,stlDT,2), 1_000_000, -28.125*bps,    True

xSCD = ql.Schedule(effDT,matDT,pDfrqQ,calUJ,mFLLW,mFLLW,dtGENf,EoMf,)
jpAMT=usAMT*ujFX ; print('xSCD:',list(xSCD)) ; print(f'jpAMT:{jpAMT:,.1f}円')
# ドル,円 クーポン
usC = ql.OvernightIndexedSwap(swPAY,usAMT,xSCD,cpnRT0,dcA360,
                                      sfIX,spdRT0,lag2d).overnightLeg()
jpC = ql.OvernightIndexedSwap(swPAY,jpAMT,xSCD,cpnRT0,dcA365,
                                       tnIX,spdRT,lag2d).overnightLeg()
# ドル,円 元本
usP = (sCF(-usAMT, xSCD[0]), sCF(usAMT, xSCD[-1]) )
jpP = (sCF(-jpAMT, xSCD[0]), sCF(jpAMT, xSCD[-1]) )
usL = usP + usC ; jpL = jpP + jpC

usNPV =  ql.CashFlows.npv(usL,  sfCvHDL, stlDtCF)
jpNPV = -ql.CashFlows.npv(jpL, jBsCvHDL, stlDtCF)/ujFX                # USDベース
print(f'(USDベース) usL:{usNPV:,.3f}  jpL:{jpNPV:,.3f}  netNPV:{usNPV+jpNPV:,.3f}')

xSCD: [Date(22,1,2026), Date(22,4,2026), Date(22,7,2026), Date(22,10,2026), Date(22,1,2027), Date(22,4,2027), Date(22,7,2027), Date(22,10,2027), Date(24,1,2028)]
jpAMT:158,152,800.0円
(USDベース) usL:-19.664  jpL:1.795  netNPV:-17.869


In [4]:
list(ql.OvernightIndexedSwap(swPAY,usAMT,xSCD,cpnRT0,dcA360,
                                      sfIX,spdRT0,lag2d).overnightLeg())

[<QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A1030> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A0FF0> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A0E30> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A0F30> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A22F0> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A2430> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A0E70> >,
 <QuantLib.QuantLib.CashFlow; proxy of <Swig Object of type 'ext::shared_ptr< CashFlow > *' at 0x00000203E17A1170> >]

In [5]:
# (hc-usL)
dfUsL = mu.legCashFlow(usL, sfCvOBJ, dc=dcA360) ; dfDSP(dfUsL)
print(f'(hc:usL) {(dfUsL.amount * dfUsL.DF).sum():,.3f}')

,fixDate,accStart,accEnd,payDate,days,nominal,rate,spread,amount,DF
0,nan,nan,nan,2026-01-22,nan,nan,nan%,nan%,"-1,000,000.00",0.99979835
1,2026-04-21,2026-01-22,2026-04-22,2026-04-24,90,"1,000,000.00",3.671350%,0.000%,"9,178.38",0.99050788
2,2026-07-21,2026-04-22,2026-07-22,2026-07-24,91,"1,000,000.00",3.558236%,0.000%,"8,994.43",0.98168692
3,2026-10-21,2026-07-22,2026-10-22,2026-10-26,92,"1,000,000.00",3.395616%,0.000%,"8,677.69",0.97307261
4,2027-01-21,2026-10-22,2027-01-22,2027-01-26,92,"1,000,000.00",3.270839%,0.000%,"8,358.81",0.96501020
5,2027-04-21,2027-01-22,2027-04-22,2027-04-26,90,"1,000,000.00",3.252120%,0.000%,"8,130.30",0.95722766
6,2027-07-21,2027-04-22,2027-07-22,2027-07-26,91,"1,000,000.00",3.252266%,0.000%,"8,221.01",0.94941440
7,2027-10-21,2027-07-22,2027-10-22,2027-10-26,92,"1,000,000.00",3.329334%,0.000%,"8,508.30",0.94140465
8,nan,nan,nan,2028-01-24,nan,nan,nan%,nan%,"1,000,000.00",0.93363441
9,2028-01-21,2027-10-22,2028-01-24,2028-01-26,94,"1,000,000.00",3.329641%,0.000%,"8,694.06",0.93345441


(hc:usL) -19.664


In [6]:
# (hc-jpL)
dfJpL = mu.legCashFlow(jpL, jBsCvOBJ, dc=dcA365) ; dfDSP(dfJpL)
print(f'(hc:jpL) {(dfJpL.amount * dfJpL.DF).sum()/ujFX:,.3f}')

,fixDate,accStart,accEnd,payDate,days,nominal,rate,spread,amount,DF
0,nan,nan,nan,2026-01-22,nan,nan,nan%,nan%,"-158,152,800.00",0.99996706
1,2026-04-21,2026-01-22,2026-04-22,2026-04-24,90,"158,152,800.00",0.460150%,-0.281%,"179,442.77",0.99852984
2,2026-07-21,2026-04-22,2026-07-22,2026-07-24,91,"158,152,800.00",0.609825%,-0.281%,"240,453.28",0.99678399
3,2026-10-21,2026-07-22,2026-10-22,2026-10-26,92,"158,152,800.00",0.739000%,-0.281%,"294,588.98",0.99473582
4,2027-01-21,2026-10-22,2027-01-22,2027-01-26,92,"158,152,800.00",0.877171%,-0.281%,"349,668.02",0.99263352
5,2027-04-21,2027-01-22,2027-04-22,2027-04-26,90,"158,152,800.00",0.991151%,-0.281%,"386,515.12",0.99014354
6,2027-07-21,2027-04-22,2027-07-22,2027-07-26,91,"158,152,800.00",1.111188%,-0.281%,"438,140.12",0.98761360
7,2027-10-21,2027-07-22,2027-10-22,2027-10-26,92,"158,152,800.00",1.266796%,-0.281%,"504,985.13",0.98465358
8,nan,nan,nan,2028-01-24,nan,nan,nan%,nan%,"158,152,800.00",0.98176649
9,2028-01-21,2027-10-22,2028-01-24,2028-01-26,94,"158,152,800.00",1.266862%,-0.281%,"515,989.77",0.98169417


(hc:jpL) -1.795


**MtM型**

In [7]:
# リセット元本の計算例
ujFX0  = ujFX*jBsCvOBJ.discount(jDT(2026,1,22))/ sfCvOBJ.discount(jDT(2026,1,22))
ujFwd1 = ujFX0*sfCvOBJ.discount(jDT(2026,4,22))/jBsCvOBJ.discount(jDT(2026,4,22))
print(f'T+0為替: {ujFX0:.4f}   4/22為替:{ujFwd1:.4f}   '
      f'new元本: {158152800/ujFwd1:,.2f}'          )

T+0為替: 158.1795   4/22為替:156.9343   new元本: 1,007,764.40


In [8]:
# MtT, JPY-Payer
effDT,   matDT,              usAMT,     spdRT,       stlDtCF     =\
stlDT, adY(calUJ,stlDT,2), 1_000_000, -28.125*bps,    True

xSCD = ql.Schedule(effDT,matDT,pDfrqQ,calUJ,mFLLW,mFLLW,dtGENf,EoMf,)
jpAMT=usAMT*ujFX ; print('xSCD:',list(xSCD)) ; print(f'jpAMT:{jpAMT:,.1f}円')
# MtM額面 算出
ujFX0 =  ujFX*jBsCvOBJ.discount(xSCD[0])/sfCvOBJ.discount(xSCD[0])
ujFwd = [ujFX0*sfCvOBJ.discount(dd)    /jBsCvOBJ.discount(dd)
         																				for dd in list(xSCD)[1:-1]]
mtmAMT = np.r_[usAMT,jpAMT/nA(ujFwd)]; print('mtmAMT:',mtmAMT)
# ドル,円 クーポン
usC = ql.OvernightIndexedSwap(swPAY,mtmAMT,xSCD,cpnRT0,dcA360,
                                      sfIX,spdRT0,lag2d).overnightLeg()
jpC = ql.OvernightIndexedSwap(swPAY,jpAMT,xSCD,cpnRT0,dcA365,
                                       tnIX,spdRT,lag2d).overnightLeg()
# ドル元本
usP = [sCF(-mtmAMT[0], xSCD[0]), sCF(mtmAMT[-1], xSCD[-1])]
for ii in range(1, len(mtmAMT)):
    usP.append(sCF(+mtmAMT[ii-1], xSCD[ii]))     #旧元本の受取り
    usP.append(sCF(-mtmAMT[ii],   xSCD[ii]))     #新元本の支払い
# 円元本
jpP = (sCF(-jpAMT, xSCD[0]), sCF(jpAMT, xSCD[-1]) )
usL = tuple(usP) + usC ; jpL = jpP + jpC

usNPV =  ql.CashFlows.npv(usL,  sfCvHDL, stlDtCF)
jpNPV = -ql.CashFlows.npv(jpL, jBsCvHDL, stlDtCF)/ujFX
print(f'(USDベース) usL:{usNPV:,.3f}  jpL:{jpNPV:,.3f}  netNPV:{usNPV+jpNPV:,.3f}')

xSCD: [Date(22,1,2026), Date(22,4,2026), Date(22,7,2026), Date(22,10,2026), Date(22,1,2027), Date(22,4,2027), Date(22,7,2027), Date(22,10,2027), Date(24,1,2028)]
jpAMT:158,152,800.0円
mtmAMT: [1000000.      1007764.39567 1015058.41216 1021811.39434 1028196.06308
 1033955.44685 1039811.62498 1045515.67096]
(USDベース) usL:-20.152  jpL:1.795  netNPV:-18.357


In [9]:
# (hc-usL)
dfUsL = mu.legCashFlow(usL, sfCvOBJ, dc=dcA360) ; dfDSP(dfUsL,8)
print(f'(hc:usL) {(dfUsL.amount * dfUsL.DF).sum():,.3f}')

,fixDate,accStart,accEnd,payDate,days,nominal,rate,spread,amount,DF
0,nan,nan,nan,2026-01-22,nan,nan,nan%,nan%,"-1,000,000.00",0.99979835
1,nan,nan,nan,2026-04-22,nan,nan,nan%,nan%,"-1,007,764.40",0.99070529
2,nan,nan,nan,2026-04-22,nan,nan,nan%,nan%,"1,000,000.00",0.99070529
3,2026-04-21,2026-01-22,2026-04-22,2026-04-24,90,"1,000,000.00",3.671350%,0.000%,"9,178.38",0.99050788
4,nan,nan,nan,2026-07-22,nan,nan,nan%,nan%,"-1,015,058.41",0.98187389
5,nan,nan,nan,2026-07-22,nan,nan,nan%,nan%,"1,007,764.40",0.98187389
6,2026-07-21,2026-04-22,2026-07-22,2026-07-24,91,"1,007,764.40",3.558236%,0.000%,"9,064.27",0.98168692
7,nan,nan,nan,2026-10-22,nan,nan,nan%,nan%,"-1,021,811.39",0.97342680
16,nan,nan,nan,2027-07-22,nan,nan,nan%,nan%,"1,033,955.45",0.94976419
17,nan,nan,nan,2027-07-22,nan,nan,nan%,nan%,"-1,039,811.62",0.94976419


(hc:usL) -20.152


In [10]:
# (hc-jpL)
dfJpL = mu.legCashFlow(jpL, jBsCvOBJ, dc=dcA365) ; dfDSP(dfJpL)
print(f'(hc:jpL) {(dfJpL.amount * dfJpL.DF).sum()/ujFX:,.3f}')

,fixDate,accStart,accEnd,payDate,days,nominal,rate,spread,amount,DF
0,nan,nan,nan,2026-01-22,nan,nan,nan%,nan%,"-158,152,800.00",0.99996706
1,2026-04-21,2026-01-22,2026-04-22,2026-04-24,90,"158,152,800.00",0.460150%,-0.281%,"179,442.77",0.99852984
2,2026-07-21,2026-04-22,2026-07-22,2026-07-24,91,"158,152,800.00",0.609825%,-0.281%,"240,453.28",0.99678399
3,2026-10-21,2026-07-22,2026-10-22,2026-10-26,92,"158,152,800.00",0.739000%,-0.281%,"294,588.98",0.99473582
4,2027-01-21,2026-10-22,2027-01-22,2027-01-26,92,"158,152,800.00",0.877171%,-0.281%,"349,668.02",0.99263352
5,2027-04-21,2027-01-22,2027-04-22,2027-04-26,90,"158,152,800.00",0.991151%,-0.281%,"386,515.12",0.99014354
6,2027-07-21,2027-04-22,2027-07-22,2027-07-26,91,"158,152,800.00",1.111188%,-0.281%,"438,140.12",0.98761360
7,2027-10-21,2027-07-22,2027-10-22,2027-10-26,92,"158,152,800.00",1.266796%,-0.281%,"504,985.13",0.98465358
8,nan,nan,nan,2028-01-24,nan,nan,nan%,nan%,"158,152,800.00",0.98176649
9,2028-01-21,2027-10-22,2028-01-24,2028-01-26,94,"158,152,800.00",1.266862%,-0.281%,"515,989.77",0.98169417


(hc:jpL) -1.795
